### Sensitivity Analysis

In [6]:
import os
import math
import warnings
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import chi2_contingency, rankdata


# =========================================================
# Config
# =========================================================
IN_PATH  = "../../Results/Data/phenotype_classification.csv"
OUT_DIR  = "../../Results/Plots/parameter_sensitivity_outputs"  
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Canonical metric columns you'd like to analyze (gracefully skipped if absent)
METRIC_CANDIDATES: Dict[str, Tuple[str, ...]] = {
    "Invasive Area":      ("Invasive Area", "Invasion Area", "Area_Invasive", "InvasiveArea"),
    "Infiltrative Area":  ("Infiltrative Area", "Infiltration Area", "Area_Infiltrative", "InfiltrativeArea"),
    "Single Defects":     ("Single Defects", "Single_Defects", "Singles", "SingleDefects"),
    "Fingers":            ("Fingers", "Fingers Count", "NumFingers", "FingerCount"),
    # "Detached Cells":     ("Detached Cells", "Detached_Cells", "Singles Detached", "Detached"),
    "Clusters":           ("Clusters", "Cluster Count", "NumClusters", "ClusterCount"),
}


PARAM_NAME_MAP: Dict[str, Tuple[str, ...]] = {
    "Contact Energy":             ("Contact Energy", "Adhesion Energy", "Jlf", "ContactEnergy", "AdhesionEnergy"),
    "Migration Coefficient":      ("Migration Coefficient", "Migration", "mu", "λ", "lambda", "Chemotaxis Lambda"),
    "Proliferative Probability":  ("Proliferative Probability", "Proliferative_Probability", "PP",
                                   "Proliferation Probability", "ProliferativeProb", "Proliferative"),
}
PHENOTYPE_CANDIDATES = ("Phenotype", "Phenotype Class", "Phenotype_Label", "Label")

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.labelsize": 18,
    "axes.labelweight": "bold",
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "font.weight": "bold",
    "legend.fontsize": 13,
})

AXIS_LABEL_WEIGHT = "bold"
TICK_LABEL_WEIGHT = "bold"
LINEWIDTH_MAIN    = 2.4


# =========================================================
# Utilities
# =========================================================
def _normalize_name(s: str) -> str:
    return str(s).strip().lower().replace(" ", "").replace("_", "")

def _find_col(df: pd.DataFrame, candidates: Tuple[str, ...]) -> Optional[str]:
    norm_map = {_normalize_name(c): c for c in df.columns}
    for c in candidates:
        key = _normalize_name(c)
        if key in norm_map:
            return norm_map[key]
    return None

def _detect_parameters(df: pd.DataFrame) -> Dict[str, str]:
    mapping = {}
    for canon, cands in PARAM_NAME_MAP.items():
        col = _find_col(df, cands)
        if col is None:
            warnings.warn(f"[WARN] Parameter '{canon}' not found (candidates={cands}). It will be skipped.")
        else:
            mapping[canon] = col
    return mapping

def _detect_metrics(df: pd.DataFrame) -> Dict[str, str]:
    m = {}
    for canon, cands in METRIC_CANDIDATES.items():
        col = _find_col(df, cands)
        if col is not None:
            m[canon] = col
    if not m:
        raise ValueError("No known metric columns were found in the file. "
                         f"Tried: {METRIC_CANDIDATES}")
    return m

def _detect_phenotype_col(df: pd.DataFrame) -> Optional[str]:
    return _find_col(df, PHENOTYPE_CANDIDATES)

def _bolden_ticklabels(ax: plt.Axes) -> None:
    for lbl in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        try:
            lbl.set_fontweight(TICK_LABEL_WEIGHT)
        except Exception:
            pass

def _savefig(fig: plt.Figure, filename: str) -> None:
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close(fig)

def distance_correlation(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    n = x.size
    if n < 3:
        return np.nan
    # distance matrices
    a = np.abs(x[:, None] - x[None, :])
    b = np.abs(y[:, None] - y[None, :])
    # double-centering
    A = a - a.mean(axis=0)[None, :] - a.mean(axis=1)[:, None] + a.mean()
    B = b - b.mean(axis=0)[None, :] - b.mean(axis=1)[:, None] + b.mean()
    dcov2_xy = (A * B).sum() / (n * n)
    dcov2_xx = (A * A).sum() / (n * n)
    dcov2_yy = (B * B).sum() / (n * n)
    if dcov2_xx <= 0 or dcov2_yy <= 0:
        return 0.0
    return float(np.sqrt(dcov2_xy) / np.sqrt(np.sqrt(dcov2_xx * dcov2_yy)))

def cramers_v_from_contingency(cont: np.ndarray) -> float:
    chi2, _, _, _ = chi2_contingency(cont)
    n = cont.sum()
    r, k = cont.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1)))) if min(r, k) > 1 else 0.0

def bin_stats(x: np.ndarray, y: np.ndarray, nbins: int = 12) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if len(x) < 3:
        return np.array([]), np.array([]), np.array([])
    bins = np.linspace(x.min(), x.max(), nbins + 1)
    idx = np.clip(np.digitize(x, bins) - 1, 0, nbins - 1)
    xc = 0.5 * (bins[:-1] + bins[1:])
    y_means = np.array([y[idx == i].mean() if np.any(idx == i) else np.nan for i in range(nbins)])
    y_sems  = np.array([y[idx == i].std(ddof=1) / np.sqrt((idx == i).sum())
                        if np.sum(idx == i) > 1 else np.nan for i in range(nbins)])
    return xc, y_means, y_sems


# =========================================================
# Load & detect columns
# =========================================================
df_raw = pd.read_csv(IN_PATH)
for c in df_raw.columns:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="ignore")

param_cols_map = _detect_parameters(df_raw)     
metric_cols_map = _detect_metrics(df_raw)      
phenotype_col = _detect_phenotype_col(df_raw)

if not param_cols_map:
    raise ValueError("No parameter columns were detected. Please check PARAM_NAME_MAP.")

df = pd.DataFrame()
for canon, col in param_cols_map.items():
    df[canon] = pd.to_numeric(df_raw[col], errors="coerce")
for canon, col in metric_cols_map.items():
    df[canon] = pd.to_numeric(df_raw[col], errors="coerce")
if phenotype_col:
    df["Phenotype"] = df_raw[phenotype_col].astype(str).str.strip()

df = df.dropna(subset=list(param_cols_map.keys())).copy()

PARAMS = list(param_cols_map.keys())  
METRICS = [m for m in metric_cols_map.keys() if m in df.columns]

# =========================================================
# 1) Correlations (Pearson, Spearman, Distance)
# =========================================================
corr_pearson  = df[PARAMS + METRICS].corr(method="pearson").loc[PARAMS, METRICS]
corr_spearman = df[PARAMS + METRICS].corr(method="spearman").loc[PARAMS, METRICS]

# Distance correlation:
distcorr = pd.DataFrame(index=PARAMS, columns=METRICS, dtype=float)
for p in PARAMS:
    for m in METRICS:
        x = df[p].to_numpy()
        y = df[m].to_numpy()
        distcorr.loc[p, m] = distance_correlation(x, y)

# Save CSVs
corr_pearson.to_csv(os.path.join(OUT_DIR, "correlation_pearson.csv"))
corr_spearman.to_csv(os.path.join(OUT_DIR, "correlation_spearman.csv"))
distcorr.to_csv(os.path.join(OUT_DIR, "correlation_distance.csv"))

# Heatmap (one PNG per matrix)
def plot_heatmap(matrix: pd.DataFrame, title: str, fname: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6.5))
    im = ax.imshow(matrix.values, cmap="coolwarm", vmin=-1, vmax=1) if "Distance" not in title \
         else ax.imshow(matrix.values, cmap="coolwarm", vmin=0, vmax=1)
    ax.set_xticks(range(matrix.shape[1])); ax.set_yticks(range(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns, rotation=30, ha="right", fontweight=TICK_LABEL_WEIGHT)
    ax.set_yticklabels(matrix.index, fontweight=TICK_LABEL_WEIGHT)
    for (i, j), val in np.ndenumerate(matrix.values):
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="black", fontsize=12, fontweight="bold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=12)
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight(TICK_LABEL_WEIGHT)
    # ax.set_title(title, pad=8, fontweight="bold")
    # ax.set_xlabel("Invasion Metrics", fontweight=AXIS_LABEL_WEIGHT)
    # ax.set_ylabel("Parameters", fontweight=AXIS_LABEL_WEIGHT)
    _bolden_ticklabels(ax)
    _savefig(fig, fname)

plot_heatmap(corr_pearson,  "Pearson Correlation",  "pearson_heatmap.png")
plot_heatmap(corr_spearman, "Spearman Correlation", "spearman_heatmap.png")
plot_heatmap(distcorr,      "Distance Correlation", "distancecorr_heatmap.png")


# =========================================================
# 2) Mutual information (nonparametric dependency)
# =========================================================
# For MI we compute per metric a 3-feature MI vector
mi_table = pd.DataFrame(index=PARAMS, columns=METRICS, dtype=float)
for m in METRICS:
    y = df[m].to_numpy()
    X = df[PARAMS].to_numpy()
    mi = mutual_info_regression(X, y, random_state=0)
    for i, p in enumerate(PARAMS):
        mi_table.loc[p, m] = mi[i]

mi_table.to_csv(os.path.join(OUT_DIR, "mutual_information.csv"))

plot_heatmap(mi_table, "Mutual Information", "mutual_info_heatmap.png")


# =========================================================
# 3) Linear regression  & Random Forest importances
# =========================================================

X = df[PARAMS].to_numpy()
Y = df[METRICS].to_numpy()

# Linear regression pipeline
lr = make_pipeline(StandardScaler(), LinearRegression())
lr.fit(X, Y)
# Average absolute coefficient across targets for each param
coef = lr.named_steps["linearregression"].coef_  # shape: (n_targets, n_features)
lr_importance = np.mean(np.abs(coef), axis=0)

# Random forest 
rf = RandomForestRegressor(n_estimators=400, random_state=1, n_jobs=-1)
rf.fit(X, Y)
rf_importance = rf.feature_importances_

importance_df = pd.DataFrame({
    "Parameter": PARAMS,
    "Linear Regression (avg |coeff|)": lr_importance,
    "Random Forest Importance": rf_importance
}).set_index("Parameter").sort_values("Random Forest Importance", ascending=False)
importance_df.to_csv(os.path.join(OUT_DIR, "feature_importance.csv"))

def bar_plot_importance(series: pd.Series, title: str, fname: str) -> None:
    fig, ax = plt.subplots(figsize=(8.2, 5.8))
    series.plot(kind="bar", ax=ax, color="#455A64")
    # ax.set_title(title, pad=8, fontweight="bold")
    # ax.set_xlabel("Parameters", fontweight=AXIS_LABEL_WEIGHT)
    ax.set_ylabel("Importance", fontweight=AXIS_LABEL_WEIGHT)
    ax.tick_params(axis="x", labelrotation=10)  

    ax.grid(axis="y", alpha=0.25)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    _bolden_ticklabels(ax)
    _savefig(fig, fname)

bar_plot_importance(importance_df["Linear Regression (avg |coeff|)"],
                    "Linear Regression Importance", "importance_linear.png")
bar_plot_importance(importance_df["Random Forest Importance"],
                    "Random Forest Feature Importance", "importance_random_forest.png")


# =========================================================
# 4) Phenotype association (χ² + Cramér’s V) 
# =========================================================
if "Phenotype" in df.columns:
    chi_rows = []
    phenos = sorted(df["Phenotype"].unique())
    pal = plt.get_cmap("Set2")
    cat_colors = [pal(i / max(1, len(phenos)-1)) for i in range(len(phenos))]

    for p in PARAMS:
        try:
            bins = pd.qcut(df[p], q=10, duplicates="drop")
        except Exception:
            bins = pd.cut(df[p], bins=10, include_lowest=True)
        cont = pd.crosstab(bins, df["Phenotype"]).reindex(columns=phenos, fill_value=0)
        chi2, pval, dof, _ = chi2_contingency(cont.values)
        V = cramers_v_from_contingency(cont.values)
        chi_rows.append({"Parameter": p, "Chi2": chi2, "p_value": pval, "DoF": dof, "CramersV": V})
       
        prop = cont.div(cont.sum(axis=1), axis=0).fillna(0.0)
        # fig, ax = plt.subplots(figsize=(10.5, 6.5))
        # bottom = np.zeros(prop.shape[0])
        # x = np.arange(prop.shape[0])
        # for i, cat in enumerate(prop.columns):
        #     ax.bar(x, prop[cat].values, bottom=bottom, color=cat_colors[i], label=str(cat), width=0.9)
        #     bottom += prop[cat].values
        # ax.set_xticks(x)
        # ax.set_xticklabels([str(c) for c in prop.index], rotation=35, ha="right", fontweight=TICK_LABEL_WEIGHT)
        # ax.set_ylabel("Proportion", fontweight=AXIS_LABEL_WEIGHT)
        # ax.set_xlabel(p, fontweight=AXIS_LABEL_WEIGHT)
        # # ax.set_title(f"Phenotype proportions by binned {p}\n"
        # #              f"χ²={chi2:.1f}, p={pval:.3g}, Cramér’s V={V:.2f}", pad=8, fontweight="bold")
        # ax.legend(frameon=True, ncol=min(4, len(phenos)))
        # ax.set_ylim(0, 1.0)
        # _bolden_ticklabels(ax)
        # _savefig(fig, f"phenotype_proportions_by_{_normalize_name(p)}.png")

    chi_df = pd.DataFrame(chi_rows).set_index("Parameter")
    chi_df.to_csv(os.path.join(OUT_DIR, "phenotype_chi_square.csv"))
else:
    warnings.warn("No phenotype column found; skipping χ² / Cramér’s V analysis.")






# =========================================================
# 6) Summary: print quick headline to console
# =========================================================
print("\nSaved outputs to:", OUT_DIR)
print("• Pearson heatmap:",        "pearson_heatmap.png")
print("• Spearman heatmap:",       "spearman_heatmap.png")
print("• DistanceCorr heatmap:",   "distancecorr_heatmap.png")
print("• Mutual information:",     "mutual_info_heatmap.png")
print("• Feature importance (CSV + PNGs):", "feature_importance.csv, importance_linear.png, importance_random_forest.png")
if "Phenotype" in df.columns:
    print("• Phenotype χ² / Cramér’s V:", "phenotype_chi_square.csv, phenotype_proportions_by_<param>.png")
print("• Metric scatter diagnostics saved for every (metric, parameter) pair.")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_75585/1893329996.py:158: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_raw[c] = pd.to_numeric(df_raw[c], errors="ignore")



Saved outputs to: ../../Results/Plots/parameter_sensitivity_outputs
• Pearson heatmap: pearson_heatmap.png
• Spearman heatmap: spearman_heatmap.png
• DistanceCorr heatmap: distancecorr_heatmap.png
• Mutual information: mutual_info_heatmap.png
• Feature importance (CSV + PNGs): feature_importance.csv, importance_linear.png, importance_random_forest.png
• Phenotype χ² / Cramér’s V: phenotype_chi_square.csv, phenotype_proportions_by_<param>.png
• Metric scatter diagnostics saved for every (metric, parameter) pair.
